In [567]:
import os
import pandas as pd
import numpy as np
import mne
import neurokit2 as nk

In [568]:
F_M_Path = r"F:\data\participants.tsv"
F_M = pd.read_csv(F_M_Path, sep="\t")
F_M

,participant_id,sex
0,sub-001,f
1,sub-002,m
2,sub-003,m
3,sub-004,m
4,sub-005,f
...,...,...
120,sub-121,m
121,sub-122,m
122,sub-123,m
123,sub-124,m


### Read ACC

In [569]:
def Read_ACC(subject_id, base_folder=r"F:\data"):
   
    subject_folder = os.path.join(base_folder, subject_id)
    acc_data = {}

    for ses in os.listdir(subject_folder):
        ses_folder = os.path.join(subject_folder, ses)
        if os.path.isdir(ses_folder) and ses.startswith("ses-"):
            acc_folder = os.path.join(ses_folder, "mov")  
            if os.path.exists(acc_folder):
                for file in os.listdir(acc_folder):
                    if file.endswith(".edf"):
                        file_path = os.path.join(acc_folder, file)
                        raw = mne.io.read_raw_edf(file_path, preload=True)
                        
                        acc_channels = [ch for ch in raw.ch_names if 'ACC X' in ch or 'ACC Y' in ch or 'ACC Z' in ch]
                        if len(acc_channels) < 3:
                            print(f"Warning: Not all ACC channels found in {file}. Found: {acc_channels}")
                        else:
                            raw.pick_channels(acc_channels)
                            acc_data[f"{ses}_{file}"] = raw
                            print(f"Loaded {file} from {ses} with channels: {acc_channels}")
    
    return acc_data

In [570]:
def ACC_Dataframe(acc_dict, subject_id, participants_file):

    participants_df = pd.read_csv(participants_file, sep='\t')
    subject_num = subject_id.replace("sub-", "")
    match = participants_df[
        participants_df['participant_id'].isin([subject_id, subject_num])]

    sex = match['sex'].values[0] if not match.empty else "Unknown"

    data_list = []
    for key, raw in acc_dict.items():
        ses_file_split = key.split("_")
        ses = ses_file_split[0]

        run = None
        for part in ses_file_split:
            if part.startswith("run-"):
                run = part.replace("run-", "")
                break

        acc_signal, times = raw.get_data(return_times=True)  
        acc_X = acc_signal[0]
        acc_Y = acc_signal[1]
        acc_Z = acc_signal[2]

        acc_magnitude = np.sqrt(acc_X**2 + acc_Y**2 + acc_Z**2)

        n_samples = acc_signal.shape[1]
        sfreq = raw.info['sfreq']

        temp_dict = {
            'subject': subject_num,
            'run': run,
            'sfreq': sfreq,
            'n_samples': n_samples,
            'signal': acc_signal,            
            'ACC_X': acc_X,
            'ACC_Y': acc_Y,
            'ACC_Z': acc_Z,
            'acc_magnitude': acc_magnitude,
            'times': times,
            'sex': sex
        }
        data_list.append(temp_dict)

    acc_df = pd.DataFrame(data_list)
    print(f"DataFrame summary created with {len(acc_df)} rows (files) for subject {subject_id} ({sex}).")
    return acc_df

### Read annotation

In [571]:
def Read_annotation(subject_id, base_folder=r"F:\data"):
    subject_folder = os.path.join(base_folder, subject_id)
    ann_dict = {}

    for ses in os.listdir(subject_folder):
        ses_folder = os.path.join(subject_folder, ses)
        if os.path.isdir(ses_folder) and ses.startswith("ses-"):
            eeg_folder = os.path.join(ses_folder, "eeg")
            if os.path.exists(eeg_folder):
                for file in os.listdir(eeg_folder):
                    if file.endswith(".tsv"):
                        file_path = os.path.join(eeg_folder, file)
                        ann = pd.read_csv(file_path, sep="\t")
                        ann_dict[f"{ses}_{file}"] = ann
                        print(f"Loaded {file} from {ses}")
    return ann_dict

In [572]:
def Annotation_Dataframe(ann_dict, subject_id):
    data_list = []
    subject_num = subject_id.replace("sub-", "")

    for key, ann in ann_dict.items():
        ses_file_split = key.split("_")
        ses = ses_file_split[0]

        run = None
        for part in ses_file_split:
            if part.startswith("run-"):
                run = part.replace("run-", "")
                break

        if all(col in ann.columns for col in ['onset', 'duration', 'eventType']):
            temp_df = pd.DataFrame({
                'subject': subject_num,
                'run': [run] * len(ann),
                'onset': ann['onset'].values,
                'duration': ann['duration'].values,
                'eventType': ann['eventType'].values,
            })
            data_list.append(temp_df)

    df_ann = pd.concat(data_list, ignore_index=True)
    print(f"Total annotations: {len(df_ann)}")
    return df_ann

### Merge

In [573]:
def merge_edf_events(edf_df, events_df):

    merged_df = pd.merge(
        edf_df,
        events_df,
        on=["subject", "run"],
        how="left"
    )

    return merged_df

### Clean


In [574]:
def remove_unwanted_events(df):
    df_clean = df[~df["eventType"].str.contains("impd|bckg", case=False, na=False)]
    return df_clean

### Processing

In [575]:
ACC_FS = 25
LOWCUT = 0.5          
HIGHCUT = 10.0        
MOTION_LPF = 10.0

In [576]:
def preprocess_acc_df(df):
   
    for idx, row in df.iterrows():
        acc_signal = np.array(row['signal']) 
        sfreq = row['sfreq']
        clean_acc = np.zeros_like(acc_signal)

        for i in range(3):
            temp = nk.signal_filter(acc_signal[i], sampling_rate=sfreq, lowcut=LOWCUT, method="butterworth")
            temp = nk.signal_filter(temp, sampling_rate=sfreq, highcut=HIGHCUT, method="butterworth")
            clean_acc[i] = temp

        signal_min = clean_acc.min(axis=1, keepdims=True)
        signal_max = clean_acc.max(axis=1, keepdims=True)
        den = signal_max - signal_min
        den[den == 0] = 1
        norm_acc = (clean_acc - signal_min) / den

        magnitude = np.sqrt(np.sum(norm_acc**2, axis=0))

        df.at[idx, 'signal'] = norm_acc
        df.at[idx, 'ACC_X'] = norm_acc[0]
        df.at[idx, 'ACC_Y'] = norm_acc[1]
        df.at[idx, 'ACC_Z'] = norm_acc[2]
        df.at[idx, 'acc_magnitude'] = magnitude

    print("ACC preprocessing complete: bandpass 0.5-10Hz + normalize + magnitude")
    return df

### Extend time & Window

In [577]:
def extend_seizure_times(df, pre=30, post=30):

    df = df.copy()
    
    df['onset_extended'] = df['onset'] - pre
    df['onset_extended'] = df['onset_extended'].apply(lambda x: max(x, 0)) 
    
    df['duration_extended'] = df['duration'] + pre + post

    print(f"Seizure times extended: +{pre}s before, +{post}s after.")
    return df

In [578]:
PRE_SEC = 600  
POST_SEC = 600

In [579]:
  def segment_acc_windows(df,
                        window_sec=60,
                        step_sec=10,
                        label_threshold=0.8,
                        use_extended=True):

    segments = []
    window_id = 0

    for (subject, run), g in df.groupby(["subject", "run"], sort=False):
        g = g.reset_index(drop=True)
        sfreq = float(g["sfreq"].iloc[0])

        for idx, row in g.iterrows():
            acc_signal = row["signal"]     
            times = row["times"]

            labels_seq = np.array(["interictal"] * acc_signal.shape[1])

            if use_extended and 'onset_extended' in row and 'duration_extended' in row:
                ictal_start = row["onset_extended"]
                ictal_end = ictal_start + row["duration_extended"]
            else:
                ictal_start = row.get("onset", None)
                ictal_end = ictal_start + row.get("duration", 0) if ictal_start is not None else None

            if ictal_start is not None and ictal_end is not None:
                for i, t in enumerate(times):
                    if ictal_start <= t < ictal_end:
                        labels_seq[i] = "ictal"
                    elif ictal_start - PRE_SEC <= t < ictal_start:
                        labels_seq[i] = "preictal"
                    elif ictal_end <= t < ictal_end + POST_SEC:
                        labels_seq[i] = "postictal"

            win_len = int(window_sec * sfreq)
            step_len = int(step_sec * sfreq)

            for start_idx in range(0, acc_signal.shape[1] - win_len + 1, step_len):
                end_idx = start_idx + win_len
                w_sig = acc_signal[:, start_idx:end_idx]
                w_times = times[start_idx:end_idx]
                w_labels = labels_seq[start_idx:end_idx]

                unique, counts = np.unique(w_labels, return_counts=True)
                ratio = dict(zip(unique, counts / len(w_labels)))

                if ratio.get("preictal", 0) >= label_threshold:
                    window_label = "preictal"
                elif ratio.get("ictal", 0) >= label_threshold:
                    window_label = "ictal"
                elif ratio.get("postictal", 0) >= label_threshold:
                    window_label = "postictal"
                else:
                    window_label = "normal"

                segments.append({
                    "window_id": window_id,
                    "subject": subject,
                    "run": run,
                    "start_time": w_times[0],
                    "end_time": w_times[-1],
                    "ACC_X": w_sig[0],
                    "ACC_Y": w_sig[1],
                    "ACC_Z": w_sig[2],
                    "acc_magnitude": np.sqrt(np.sum(w_sig**2, axis=0)),
                    "times": w_times,
                    "label": window_label,
                    "preictal_ratio": ratio.get("preictal", 0.0),
                    "ictal_ratio": ratio.get("ictal", 0.0),
                    "postictal_ratio": ratio.get("postictal", 0.0),
                    "window_length": w_sig.shape[1],
                    "sfreq": sfreq,
                    "sex": row.get("sex", "Unknown")
                })
                window_id += 1

    seg_df = pd.DataFrame(segments)
    print(f"ACC Segmentation complete: {len(seg_df)} windows created with 4 labels.")
    return seg_df

### features

In [580]:
def extract_acc_features(seg_df):

    df = seg_df.copy()

    df["activity_score"] = 0.0
    df["rms_mag"] = 0.0
    df["std_mag"] = 0.0

    for idx, row in df.iterrows():
        acc_x = np.array(row["ACC_X"], dtype=float)
        acc_y = np.array(row["ACC_Y"], dtype=float)
        acc_z = np.array(row["ACC_Z"], dtype=float)

        mag = np.sqrt(acc_x**2 + acc_y**2 + acc_z**2)

        mag_motion = row.get("mag_motion_window", mag)
        if mag_motion is None:
            mag_motion = mag

        df.at[idx, "activity_score"] = float(np.sqrt(np.mean(np.array(mag_motion)**2)))
        df.at[idx, "rms_mag"] = float(np.sqrt(np.mean(mag**2)))
        df.at[idx, "std_mag"] = float(np.std(mag))

    print(f"Extracted features for {len(df)} windows, all original columns retained.")
    return df

### Apply

In [581]:
def process_subject_ACC(subject_id, base_folder, participants_file,
                        pre_extend=30, post_extend=30,
                        window_sec=60, step_sec=10, label_threshold=0.8,
                        acc_fs=25):

    acc_data = Read_ACC(subject_id, base_folder=base_folder)
    
    acc_df = ACC_Dataframe(acc_data, subject_id, participants_file)
    
    ann = Read_annotation(subject_id, base_folder=base_folder)
    df_ann = Annotation_Dataframe(ann, subject_id)
    
    merged_df = merge_edf_events(acc_df, df_ann)
    
    cleaned_df = remove_unwanted_events(merged_df)
    
    preprocessed_df = preprocess_acc_df(cleaned_df)
    
    extended_df = extend_seizure_times(preprocessed_df, pre=pre_extend, post=post_extend)
    
    acc_segments_df = segment_acc_windows(extended_df,
                                          window_sec=window_sec,
                                          step_sec=step_sec,
                                          label_threshold=label_threshold,
                                          use_extended=True)
    
    features_df = extract_acc_features(acc_segments_df)
    
    return features_df, acc_segments_df

In [582]:
subject_id = "sub-075"
base_folder = r"F:\data"

ACC, acc_segments_df = process_subject_ACC(subject_id,base_folder,F_M_Path)

print(ACC.head())
print(acc_segments_df['label'].value_counts())


Extracting EDF parameters from F:\data\sub-075\ses-01\mov\sub-075_ses-01_task-szMonitoring_run-01_mov.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 539074  =      0.000 ... 21562.960 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loaded sub-075_ses-01_task-szMonitoring_run-01_mov.edf from ses-01 with channels: ['EEG SD ACC X', 'EEG SD ACC Y', 'EEG SD ACC Z', 'ECGEMG SD ACC X', 'ECGEMG SD ACC Y', 'ECGEMG SD ACC Z']
Extracting EDF parameters from F:\data\sub-075\ses-01\mov\sub-075_ses-01_task-szMonitoring_run-02_mov.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 523399  =      0.000 ... 20935.960 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Loaded sub-075_ses-01_task-szMonitoring_run-02_mov.edf from ses-01 with channels: ['EEG SD ACC X', 'EEG SD ACC Y', 'EEG SD ACC Z', 'ECGEMG SD ACC X', 'ECG

In [583]:
Label_counts =ACC['label'].value_counts().reset_index()
Label_counts.columns = ['Label', 'Count']
Label_counts

,Label,Count
0,normal,4055
1,preictal,114
2,postictal,114
3,ictal,17


In [584]:
ACC.to_csv(f"{subject_id}_acc_features.csv", index=False)